## Environment setup and synthetic data

# Building a Healthcare Analytics Dataset from Raw Multi-Source Data

**Important — Do not modify these datasets**

The synthetic datasets below are fixed and used by all learners.
Do not modify this cell or change any data values, as **grading is based on the exact outputs** produced from these datasets.

You should only write code in the sections of the notebook that are clearly marked for you to complete.

In [1]:
# ⚠️ DO NOT MODIFY THIS CELL
# This dataset is fixed and used for consistent grading across all learners.

import pandas as pd
import numpy as np

ehr_df = pd.DataFrame({
    "patient_id": ["P001","P002","P003","P003","P004"],
    "patient_name": ["John Doe","Jane Smith","Alice Brown","Alice Brown","Bob Lee"],
    "dob": pd.to_datetime(["1980-05-12","1975-09-30","1990-02-15","1990-02-15","1965-11-01"]),
    "encounter_id": ["E1","E2","E3","E3","E4"],
    "encounter_date": pd.to_datetime(["2023-01-10","2023-01-12","2023-01-15","2023-01-15","2023-01-18"]),
    "heart_rate": [72,-10,85,85,220]
})

claims_df = pd.DataFrame({
    "member_id": ["001","002","003","004"],
    "diagnosis_code": ["I10","E11","E11","I10"],
    "claim_amount": [12000,18000,15000,20000]
})

labs_df = pd.DataFrame({
    "lab_patient_id": ["P-001", "P-002", "P-003", "P-004"],
    "lab_date": pd.to_datetime(["2023-01-08", "2023-01-09", "2023-01-14", "2023-01-17"]),
    "glucose": [95, 5.5, 110, 6.2],
    "unit": ["mg/dL", "mmol/L", "mg/dL", "mmol/L"]
})

### Step 1: Inspect dataset structure

**What is expected**
- Display the number of rows and columns of the EHR dataset
- Display all column names and their data types
(For example, by showing the dataset shape and column data types.)

In [2]:
# Display the number of rows and columns
print("Dataset Shape:")
print(ehr_df.shape)

print("\nNumber of Rows:", ehr_df.shape[0])
print("Number of Columns:", ehr_df.shape[1])

# Display column names
print("\nColumn Names:")
print(ehr_df.columns)

# Display data types
print("\nData Types:")
print(ehr_df.dtypes)

Dataset Shape:
(5, 6)

Number of Rows: 5
Number of Columns: 6

Column Names:
Index(['patient_id', 'patient_name', 'dob', 'encounter_id', 'encounter_date',
       'heart_rate'],
      dtype='object')

Data Types:
patient_id                object
patient_name              object
dob               datetime64[ns]
encounter_id              object
encounter_date    datetime64[ns]
heart_rate                 int64
dtype: object


### Step 2: Identify duplicate records

**What is expected**
- Print the total number of duplicate rows
- Show which rows are duplicates using a clear row-level indicator (For example, by displaying a row-level indicator showing True/False for duplicates.)


In [3]:
# Identify duplicate rows
duplicates = ehr_df.duplicated()

# Count duplicate records
print("Number of duplicate records:")
print(duplicates.sum())

# Display the duplicated rows
print("\nDuplicate rows:")
print(ehr_df[duplicates])

Number of duplicate records:
1

Duplicate rows:
  patient_id patient_name        dob encounter_id encounter_date  heart_rate
3       P003  Alice Brown 1990-02-15           E3     2023-01-15          85


### Step 3: Remove duplicate records

**What is expected**
- Remove duplicate rows from the dataset
- Print the updated dataset shape (rows and columns)

In [4]:
# Remove duplicate rows
ehr_df = ehr_df.drop_duplicates()

# Display the updated dataset size
print("Updated Dataset Shape:")
print(ehr_df.shape)

Updated Dataset Shape:
(4, 6)


### Step 4: Identify invalid clinical values

**What is expected**
- Print any rows where heart rate values are invalid (less than zero)

In [5]:
# Identify invalid heart rate values
invalid_heart_rate = ehr_df[
    (ehr_df["heart_rate"] < 30) |
    (ehr_df["heart_rate"] > 220)
]

# Display invalid records
print("Invalid Heart Rate Records:")
print(invalid_heart_rate)

# Count invalid values
print("\nNumber of Invalid Heart Rate Values:")
print(len(invalid_heart_rate))

Invalid Heart Rate Records:
  patient_id patient_name        dob encounter_id encounter_date  heart_rate
1       P002   Jane Smith 1975-09-30           E2     2023-01-12         -10

Number of Invalid Heart Rate Values:
1


### Step 5: Correct invalid clinical values

**What is expected**
- Replace invalid heart rate values with missing values (NaN)
- Print the updated heart rate column

In [6]:
# Replace invalid heart rate values with NaN
ehr_df.loc[
    (ehr_df["heart_rate"] < 30) |
    (ehr_df["heart_rate"] > 220),
    "heart_rate"
] = np.nan

# Display the updated heart_rate column
print("Updated Heart Rate Values:")
print(ehr_df["heart_rate"])

Updated Heart Rate Values:
0     72.0
1      NaN
2     85.0
4    220.0
Name: heart_rate, dtype: float64


### Step 6: Standardize patient identifiers across sources

**What is expected**
- Create a standardized patient identifier column
- Print the standardized identifiers from all three datasets

In [14]:
# Standardize patient identifiers in all datasets

ehr_df["patient_id"] = (
    ehr_df["patient_id"]
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.strip()
)

claims_df["member_id"] = (
    "P" + claims_df["member_id"] # Add 'P' prefix for consistency
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.strip()
)

labs_df["lab_patient_id"] = (
    labs_df["lab_patient_id"]
    .str.upper()
    .str.replace("-", "", regex=False)
    .str.strip()
)

# Display standardized IDs
print("EHR Patient IDs")
print(ehr_df["patient_id"])

print("\nClaims Patient IDs")
print(claims_df["member_id"])

print("\nLaboratory Patient IDs")
print(labs_df["lab_patient_id"])


EHR Patient IDs
0    P001
1    P002
2    P003
4    P004
Name: patient_id, dtype: object

Claims Patient IDs
0    P001
1    P002
2    P003
3    P004
Name: member_id, dtype: object

Laboratory Patient IDs
0    P001
1    P002
2    P003
3    P004
Name: lab_patient_id, dtype: object


### Step 7: Map diagnosis codes to clinical conditions

**What is expected**
- Map diagnosis codes to readable clinical condition names
- Print diagnosis codes alongside mapped conditions

In [10]:
# Dictionary that maps diagnosis codes to clinical condition names
diagnosis_mapping = {
    "E11": "Type 2 Diabetes Mellitus",
    "I10": "Essential Hypertension",
    "J45": "Asthma"
}

# Create a new column containing readable diagnosis names in claims_df
claims_df["diagnosis_name"] = (
    claims_df["diagnosis_code"]
    .map(diagnosis_mapping)
    .fillna("Unknown Condition")
)

# Display the updated data
print(claims_df[["diagnosis_code", "diagnosis_name"]])


  diagnosis_code            diagnosis_name
0            I10    Essential Hypertension
1            E11  Type 2 Diabetes Mellitus
2            E11  Type 2 Diabetes Mellitus
3            I10    Essential Hypertension


### Step 8: Resolve laboratory unit mismatches

**What is expected**
- Convert all glucose values to a consistent unit (mg/dL)
- Print original values, units, and converted values

In [11]:
# Define conversion factor: 1 mmol/L = 18.018 mg/dL
mmol_to_mgdl_factor = 18.018

# Create a new column for converted glucose values, initialized with original values
labs_df['glucose_mg_dL'] = labs_df['glucose']

# Convert glucose values from mmol/L to mg/dL where unit is 'mmol/L'
labs_df.loc[labs_df['unit'] == 'mmol/L', 'glucose_mg_dL'] = \
    labs_df.loc[labs_df['unit'] == 'mmol/L', 'glucose'] * mmol_to_mgdl_factor

# Print original values, units, and converted values
print("Original Glucose | Unit   | Converted Glucose (mg/dL)")
print("----------------------------------------------------")
for index, row in labs_df.iterrows():
    print(f"{row['glucose']:<16.2f} | {row['unit']:<6} | {row['glucose_mg_dL']:.2f}")


Original Glucose | Unit   | Converted Glucose (mg/dL)
----------------------------------------------------
95.00            | mg/dL  | 95.00
5.50             | mmol/L | 99.10
110.00           | mg/dL  | 110.00
6.20             | mmol/L | 111.71


### Step 9: Merge EHR, claims, and laboratory data

**What is expected**
- Merge all datasets using the standardized patient identifier
- Display the column names to confirm the merge
- Display key fields from each source system (for example, a patient identifier, an encounter field, a diagnosis field, and a laboratory value) to verify the merged result

In [15]:
# Merge EHR with Claims
merged_df = pd.merge(
    ehr_df,
    claims_df,
    left_on="patient_id",
    right_on="member_id",
    how="inner"
)

# Merge the result with Laboratory data
merged_df = pd.merge(
    merged_df,
    labs_df,
    left_on="patient_id",
    right_on="lab_patient_id",
    how="inner"
)

# Display merged dataset
print("Merged Dataset:")
display(merged_df.head())

# Display dataset size
print("\nMerged Dataset Shape:")
print(merged_df.shape)


Merged Dataset:


,patient_id,patient_name,dob,encounter_id,encounter_date,heart_rate,member_id,diagnosis_code,claim_amount,diagnosis_name,lab_patient_id,lab_date,glucose,unit,glucose_mg_dL
0,P001,John Doe,1980-05-12,E1,2023-01-10,72.0,P001,I10,12000,Essential Hypertension,P001,2023-01-08,95.0,mg/dL,95.0000
1,P002,Jane Smith,1975-09-30,E2,2023-01-12,NaN,P002,E11,18000,Type 2 Diabetes Mellitus,P002,2023-01-09,5.5,mmol/L,99.0990
2,P003,Alice Brown,1990-02-15,E3,2023-01-15,85.0,P003,E11,15000,Type 2 Diabetes Mellitus,P003,2023-01-14,110.0,mg/dL,110.0000
3,P004,Bob Lee,1965-11-01,E4,2023-01-18,220.0,P004,I10,20000,Essential Hypertension,P004,2023-01-17,6.2,mmol/L,111.7116



Merged Dataset Shape:
(4, 15)


### Step 10: Apply HIPAA safe harbor de-identification

**What is expected**
- Remove the patient name column, which represents a direct identifier in this dataset
- Print the column list to confirm that the patient_name column has been removed
- Create a generalized year-of-birth field from the date of birth (for example, extracting the year)
- To keep the output readable for grading, **print only the patient identifier and birth year columns**

Note: The full dataset should still be updated. You are only limiting what is printed.

In [24]:
# Create a generalized birth year from the date of birth
if "dob" in merged_df.columns:
    merged_df["birth_year"] = pd.to_datetime(
        merged_df["dob"]
    ).dt.year
else:
    print("'dob' column not found, 'birth_year' cannot be created from it.")

# Remove direct identifiers
columns_to_drop = []
if "patient_name" in merged_df.columns:
    columns_to_drop.append("patient_name")
if "dob" in merged_df.columns:
    columns_to_drop.append("dob")

if columns_to_drop:
    merged_df = merged_df.drop(
        columns=columns_to_drop
    )
else:
    print("No columns to drop ('patient_name', 'dob') found.")

# Print the column list to confirm that the patient_name column has been removed
print("Updated Columns after de-identification:")
print(merged_df.columns.tolist())

# Display only the patient identifier and birth year columns
print("\nPatient ID and Birth Year:")
# Check if 'birth_year' exists before displaying
if "birth_year" in merged_df.columns:
    display(merged_df[["patient_id", "birth_year"]].head())
else:
    print("'birth_year' column not available for display.")

'dob' column not found, 'birth_year' cannot be created from it.
No columns to drop ('patient_name', 'dob') found.
Updated Columns after de-identification:
['patient_id', 'encounter_id', 'encounter_date', 'heart_rate', 'member_id', 'diagnosis_code', 'claim_amount', 'diagnosis_name', 'lab_patient_id', 'lab_date', 'glucose', 'unit', 'glucose_mg_dL', 'birth_year']

Patient ID and Birth Year:


,patient_id,birth_year
0,P001,1980
1,P002,1975
2,P003,1990
3,P004,1965


### Step 11: Create encounter utilization features

**What is expected**
- Count the number of encounters per patient
- Print the encounter count for each patient using the patient identifier

In [25]:
# Count the number of encounters for each patient
encounter_features = (
    merged_df
    .groupby("patient_id")["encounter_id"]
    .count()
    .reset_index(name="encounter_count")
)

# Display the encounter utilization feature
print("Encounter Utilization Features:")
display(encounter_features)

Encounter Utilization Features:


,patient_id,encounter_count
0,P001,1
1,P002,1
2,P003,1
3,P004,1


### Step 12: Engineer final model-ready features

**What is expected**
- Aggregate patient-level features
- Create a final feature table with one row per patient that includes a glucose-related feature and an encounter count
- Print the final feature table

In [ ]:
# your code goes here

In [27]:


# Aggregate glucose-related features for each patient
glucose_features = merged_df.groupby("patient_id")["glucose_mg_dL"].mean().reset_index(name="avg_glucose_mg_dL")

# Merge with encounter features
final_feature_table = pd.merge(
    glucose_features,
    encounter_features,
    on="patient_id",
    how="inner"
)

# Print the final feature table
print("Final Model-Ready Feature Table:")
display(final_feature_table)

Final Model-Ready Feature Table:


,patient_id,avg_glucose_mg_dL,encounter_count
0,P001,95.0000,1
1,P002,99.0990,1
2,P003,110.0000,1
3,P004,111.7116,1
